# EDA and Cleaning: University Syllabus Dataset

## 1. EDA를 새로 시작하는 이유

기존 EDA 결과는 데이터가 너무 많고 중복이 많아 만족스럽지 않았다. 컴퓨터공학 / AI / 데이터 중심 커리큘럼 생성에 직접 필요하지 않은 과목이 많이 섞였고, 건축, 토목, 기계, 재료, 환경 등 목표 도메인과 거리가 있는 공학 계열 과목도 많이 남았다.

이번 EDA는 원본 데이터에서 다시 시작한다. 데이터 양보다 데이터 품질을 우선하고, 목표 도메인과 관련성이 약한 데이터는 과감히 제거한다. 최종 목표는 RAG 또는 커리큘럼 생성 로직에서 참고할 수 있는 고품질 강의계획서 데이터셋을 만드는 것이다.

이번 수정에서는 `learning_objective`와 `main_textbook`을 핵심 품질 컬럼으로 본다. 두 컬럼은 과목의 학습 목표와 학습 자료를 이해하는 데 중요하므로, 최종 데이터에는 두 컬럼이 모두 실질적으로 존재하는 행만 남긴다. 반면 `sub_textbook`, `reference_material`, `prerequisite_material`은 보조 정보라 비어 있어도 단독 제거 기준으로 사용하지 않는다.

노트북과 산출물은 계속 작업하던 `curriculum-data/raw/university_syllabus/` 하위에서 관리한다. 모든 CSV 산출물은 `curriculum-data/raw/university_syllabus/data/`에 저장한다.

## 2. 데이터 로드 및 기본 구조 확인

원본 CSV를 로드하고 기본 구조를 확인한다. 이 단계에서는 아직 행을 제거하지 않는다. shape, 컬럼명, dtype, 결측치, 고유값 개수, 완전 중복 행 수, 샘플 5개를 출력한다.

In [ ]:
from pathlib import Path
import html
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / ".git").exists() or (path / "curriculum-data").exists():
            return path
    return start


ROOT = find_project_root(Path.cwd())
SYLLABUS_DIR = ROOT / "curriculum-data" / "raw" / "university_syllabus"
OUTPUT_DIR = SYLLABUS_DIR / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE_CANDIDATES = [
    "한국교육학술정보원_대학학과커리큘럼_강의계획서_20221227.csv",
    "한국교육학술정보원_대학학과커리큘럼_강의계획서_20221227 (1).csv",
]
SEARCH_DIRS = [ROOT / "data" / "raw", ROOT / "data", ROOT, SYLLABUS_DIR]
ENCODING_CANDIDATES = ["cp949", "utf-8-sig", "utf-8"]

eda_logs = []


def add_log(step, rows_before, rows_after, cols_before, cols_after, note):
    rows_removed = 0 if rows_before == 0 else rows_before - rows_after
    removal_ratio = 0 if rows_before == 0 else round(rows_removed / rows_before * 100, 2)
    eda_logs.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_removed,
        "removal_ratio": removal_ratio,
        "cols_before": cols_before,
        "cols_after": cols_after,
        "note": note,
    })


def find_source_csv() -> Path:
    checked = []
    for directory in SEARCH_DIRS:
        for filename in SOURCE_FILE_CANDIDATES:
            path = directory / filename
            checked.append(path)
            if path.exists():
                return path
    # 파일명이 조금 달라져도 20221227 원본 CSV를 찾을 수 있게 보조 탐색을 둔다.
    for directory in SEARCH_DIRS:
        if directory.exists():
            matches = sorted(directory.glob("*20221227*.csv"))
            if matches:
                return matches[0]
    checked_text = "\n".join(map(str, checked))
    raise FileNotFoundError("원본 CSV 파일을 찾지 못했습니다. 확인 경로:\n" + checked_text)


def read_csv_with_fallback(path: Path):
    errors = []
    for encoding in ENCODING_CANDIDATES:
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str, low_memory=False), encoding
        except UnicodeDecodeError as exc:
            errors.append(f"{encoding}: {exc}")
    raise UnicodeDecodeError("csv", b"", 0, 1, " / ".join(errors))


source_path = find_source_csv()
raw_df, source_encoding = read_csv_with_fallback(source_path)

print(f"Loaded source path: {source_path.resolve()}")
print(f"Detected encoding: {source_encoding}")

shape_df = pd.DataFrame({"metric": ["rows", "columns"], "value": [raw_df.shape[0], raw_df.shape[1]]})
schema_df = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": raw_df.dtypes.astype(str).values,
    "non_null_count": raw_df.notna().sum().values,
    "missing_count": raw_df.isna().sum().values,
    "missing_ratio": (raw_df.isna().mean() * 100).round(2).values,
    "unique_count": raw_df.nunique(dropna=True).values,
})
missing_df = schema_df[["column", "missing_count", "missing_ratio"]].sort_values("missing_ratio", ascending=False)
duplicate_summary_df = pd.DataFrame({"metric": ["complete_duplicate_rows"], "value": [int(raw_df.duplicated().sum())]})

add_log("load_raw_data", 0, len(raw_df), 0, raw_df.shape[1], f"Loaded {source_path} with encoding={source_encoding}")

display(shape_df)
display(schema_df)
display(missing_df)
display(duplicate_summary_df)
display(raw_df.head())


## 3. 컬럼명 정리 및 원본 추적 컬럼 추가

원본 한글 컬럼명을 영어 snake_case로 변경한다. `source_row_number`는 정제 후에도 원본 행을 추적하기 위한 컬럼이다. 중복 제거 기준에는 포함하지 않지만, 검토와 디버깅을 위해 최종 데이터에는 유지한다.

In [ ]:
COLUMN_RENAME_MAP = {
    "연도": "year",
    "대학교명": "university_name",
    "단과대학명": "college_name",
    "학부 과명": "department_name",
    "과목명": "course_name",
    "학년": "grade",
    "학기": "semester",
    "학점": "credit",
    "이론시간": "lecture_hours",
    "실습시간": "practice_hours",
    "과목구분": "course_type",
    "학습목표": "learning_objective",
    "주교재": "main_textbook",
    "부교재": "sub_textbook",
    "참고자료": "reference_material",
    "선행학습자료": "prerequisite_material",
}

missing_source_columns = [col for col in COLUMN_RENAME_MAP if col not in raw_df.columns]
if missing_source_columns:
    print("[WARN] 원본 데이터에 없는 컬럼:", missing_source_columns)

rows_before, cols_before = raw_df.shape
df = raw_df.copy().rename(columns={k: v for k, v in COLUMN_RENAME_MAP.items() if k in raw_df.columns})
add_log("rename_columns", rows_before, len(df), cols_before, df.shape[1], "Renamed available Korean columns to English names.")

rows_before, cols_before = df.shape
df.insert(0, "source_row_number", raw_df.index + 2)
add_log("add_source_row_number", rows_before, len(df), cols_before, df.shape[1], "Added source_row_number as original index + 2.")

schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_ratio": (df.isna().mean() * 100).round(2).values,
    "unique_count": df.nunique(dropna=True).values,
})
display(schema_df)


## 4. 컬럼 역할 정의

커리큘럼 생성에 직접 쓰는 컬럼은 `course_name`, `grade`, `semester`, `credit`, `learning_objective`, `prerequisite_material`, `main_textbook`이다. `course_type`은 앞선 노이즈 제거에는 필요하지만, 최종 추천용 데이터셋에서는 제거한다.

메타데이터로 유지할 컬럼은 `source_row_number`, `university_name`, `college_name`, `department_name`이다. 보조 정보 컬럼인 `sub_textbook`, `reference_material`, `prerequisite_material`은 비어 있을 수 있으므로 단독 제거 기준으로 사용하지 않는다.

In [ ]:
column_roles = {
    "curriculum_generation": ["course_name", "grade", "semester", "credit", "learning_objective", "prerequisite_material", "main_textbook"],
    "metadata_keep": ["source_row_number", "university_name", "college_name", "department_name"],
    "supporting_information": ["sub_textbook", "reference_material"],
    "used_for_cleaning_only": ["course_type", "lecture_hours", "practice_hours"],
}
column_role_df = pd.DataFrame(
    [{"role": role, "column": col, "exists": col in df.columns} for role, cols in column_roles.items() for col in cols]
)
display(column_role_df)


## 5. 텍스트 정규화

중복 제거 전에 텍스트 정규화가 필요하다. 눈에 보이지 않는 공백, 줄바꿈, 탭, full-width 공백, HTML entity 때문에 동일한 강의계획서가 서로 다른 값으로 인식될 수 있기 때문이다.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.replace("\u3000", " ")
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


rows_before, cols_before = df.shape
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
df_normalized = df.copy()
for col in text_columns:
    df_normalized[col] = df_normalized[col].map(normalize_text)

changed_rows = (df[text_columns].fillna("").astype(str) != df_normalized[text_columns]).any(axis=1).sum() if text_columns else 0
df = df_normalized.copy()
add_log("normalize_text", rows_before, len(df), cols_before, df.shape[1], f"Normalized text columns. Changed rows={changed_rows}.")

display(pd.DataFrame({"normalized_text_columns": text_columns}))
display(pd.DataFrame({"changed_rows": [int(changed_rows)]}))


## 6. 목표 도메인 엄격 필터링

도메인 필터링 조건은 유지한다. 목표 학과와 목표 과목이 함께 매칭되는 경우, 목표 학과 안의 기초/핵심 과목인 경우, 또는 목표 학과 밖이라도 강한 AI/데이터/컴공 과목인 경우만 유지한다.

In [ ]:
TARGET_DEPARTMENT_KEYWORDS = [
    "컴퓨터", "소프트웨어", "인공지능", "AI", "데이터", "데이터사이언스", "전산", "정보보호", "정보보안", "사이버보안", "사이버", "보안", "컴퓨터정보", "컴퓨터소프트웨어", "소프트웨어융합",
    "정보통신", "전자", "전기", "반도체", "임베디드", "IoT", "로봇", "제어", "제어계측", "통신", "전자정보",
]
TARGET_COURSE_KEYWORDS = [
    "프로그래밍", "코딩", "파이썬", "Python", "자바", "Java", "C언어", "C++", "객체지향", "자료구조", "알고리즘", "운영체제", "컴퓨터구조", "컴퓨터아키텍처", "데이터베이스", "DB", "네트워크", "소프트웨어공학", "웹", "앱", "모바일", "서버", "클라우드",
    "인공지능", "AI", "머신러닝", "기계학습", "딥러닝", "빅데이터", "데이터사이언스", "데이터분석", "데이터마이닝", "자연어처리", "NLP", "컴퓨터비전", "영상처리", "추천시스템", "강화학습",
    "정보보호", "정보보안", "보안", "암호", "해킹", "사이버보안",
    "정보통신", "통신", "무선통신", "디지털통신", "신호처리", "신호및시스템", "회로", "회로이론", "전기회로", "전자회로", "논리회로", "디지털논리", "디지털시스템", "마이크로프로세서", "마이크로컨트롤러", "임베디드", "IoT", "센서", "반도체", "반도체소자", "반도체공정", "집적회로", "VLSI", "FPGA", "제어공학", "자동제어", "로봇공학", "메카트로닉스",
]
FOUNDATIONAL_COURSE_KEYWORDS = ["선형대수", "이산수학", "확률", "통계", "확률및통계", "수치해석", "공학수학", "캡스톤", "프로젝트", "설계", "실험"]
STRONG_AI_CS_COURSE_KEYWORDS = [
    "인공지능", "AI", "머신러닝", "기계학습", "딥러닝", "빅데이터", "데이터사이언스", "데이터분석", "데이터마이닝", "자연어처리", "NLP", "컴퓨터비전", "추천시스템", "강화학습", "알고리즘", "운영체제", "데이터베이스", "컴퓨터구조", "정보보호", "정보보안", "사이버보안", "암호", "해킹",
]
EXCLUDED_DEPARTMENT_KEYWORDS = ["건축", "토목", "도시", "기계", "화학", "환경", "재료", "신소재", "섬유", "식품", "생명", "바이오", "간호", "의학", "약학", "농업", "산림", "조경", "경영", "경제", "행정", "법", "인문", "사회", "교육", "예술", "체육"]


def contains_any(series: pd.Series, keywords: list[str]) -> pd.Series:
    text = series.fillna("").astype(str).str.casefold()
    result = pd.Series(False, index=series.index)
    for keyword in keywords:
        result = result | text.str.contains(keyword.casefold(), regex=False, na=False)
    return result


rows_before, cols_before = df.shape
df_filtered_base = df.copy()
df_filtered_base["target_department_match"] = contains_any(df_filtered_base.get("department_name", pd.Series("", index=df_filtered_base.index)), TARGET_DEPARTMENT_KEYWORDS)
df_filtered_base["target_course_match"] = contains_any(df_filtered_base.get("course_name", pd.Series("", index=df_filtered_base.index)), TARGET_COURSE_KEYWORDS)
df_filtered_base["foundational_course_match"] = contains_any(df_filtered_base.get("course_name", pd.Series("", index=df_filtered_base.index)), FOUNDATIONAL_COURSE_KEYWORDS)
df_filtered_base["strong_ai_cs_course_match"] = contains_any(df_filtered_base.get("course_name", pd.Series("", index=df_filtered_base.index)), STRONG_AI_CS_COURSE_KEYWORDS)
df_filtered_base["excluded_department_match"] = contains_any(df_filtered_base.get("department_name", pd.Series("", index=df_filtered_base.index)), EXCLUDED_DEPARTMENT_KEYWORDS)

condition_a = df_filtered_base["target_department_match"] & df_filtered_base["target_course_match"]
condition_b = df_filtered_base["target_department_match"] & df_filtered_base["foundational_course_match"]
condition_c = ~df_filtered_base["target_department_match"] & df_filtered_base["strong_ai_cs_course_match"]
df_filtered_base["final_target_keep"] = condition_a | condition_b | condition_c
df_filtered_base["keep_reason"] = ""
df_filtered_base.loc[condition_a, "keep_reason"] = "target_department_and_target_course"
df_filtered_base.loc[condition_b & ~condition_a, "keep_reason"] = "target_department_and_foundational_course"
df_filtered_base.loc[condition_c, "keep_reason"] = "strong_ai_cs_course_outside_target_department"

df_step_01 = df_filtered_base.loc[df_filtered_base["final_target_keep"]].copy()
step_01_path = OUTPUT_DIR / "step_01_target_domain_filtered.csv"
df_step_01.to_csv(step_01_path, index=False, encoding="utf-8-sig")
print(f"Saved: {step_01_path.resolve()}")

add_log("target_domain_strict_filtering", rows_before, len(df_step_01), cols_before, df_step_01.shape[1], "Kept only strict target-domain rows using conditions A/B/C.")

display(pd.DataFrame({
    "metric": ["raw_rows", "final_keep_rows", "removed_rows", "final_keep_ratio"],
    "value": [len(df_filtered_base), len(df_step_01), len(df_filtered_base) - len(df_step_01), round(len(df_step_01) / len(df_filtered_base) * 100, 2)],
}))
display(df_step_01["keep_reason"].value_counts().rename_axis("keep_reason").reset_index(name="row_count"))


## 7. 교양 및 명백한 노이즈 과목 제거

도메인 필터링 결과에서 명백히 커리큘럼 지식으로 쓰기 어려운 교양, 진로, 취업, 창업, 현장실습류 과목을 제거한다.

In [ ]:
NOISE_COURSE_KEYWORDS = ["진로", "진로설계", "취업", "창업", "인성", "봉사", "리더십", "글쓰기", "영어", "의사소통", "체육", "예술", "현장실습", "국외현장실습", "자율형현장실습", "표준형현장실습"]

rows_before, cols_before = df_step_01.shape
df_noise_base = df_step_01.copy()
general_education_mask = df_noise_base.get("course_type", pd.Series("", index=df_noise_base.index)).fillna("").astype(str).str.contains("교양", regex=False, na=False)
noise_course_mask = contains_any(df_noise_base.get("course_name", pd.Series("", index=df_noise_base.index)), NOISE_COURSE_KEYWORDS)
remove_noise_mask = general_education_mask | noise_course_mask

df_step_02 = df_noise_base.loc[~remove_noise_mask].copy()
step_02_path = OUTPUT_DIR / "step_02_noise_removed.csv"
df_step_02.to_csv(step_02_path, index=False, encoding="utf-8-sig")
print(f"Saved: {step_02_path.resolve()}")

add_log("remove_general_education_and_noise_courses", rows_before, len(df_step_02), cols_before, df_step_02.shape[1], "Removed general education and obvious non-curricular noise courses.")
display(pd.DataFrame({
    "metric": ["rows_before", "general_education_rows", "noise_course_rows", "rows_removed", "rows_after", "removal_ratio"],
    "value": [rows_before, int(general_education_mask.sum()), int(noise_course_mask.sum()), int(remove_noise_mask.sum()), len(df_step_02), round(remove_noise_mask.sum() / rows_before * 100, 2) if rows_before else 0],
}))


## 8. 중복 제거

중복 제거 조건은 강의계획서 내용에 가까운 컬럼만 사용한다. `source_row_number`, 학교/학과 메타데이터, 필터링 플래그는 제외한다.

In [ ]:
df_dedup_base = df_step_02.copy()
for col in df_dedup_base.select_dtypes(include=["object", "string"]).columns:
    df_dedup_base[col] = df_dedup_base[col].map(normalize_text)

DEDUP_CANDIDATE_COLUMNS = ["course_name", "grade", "semester", "credit", "course_type", "learning_objective", "main_textbook", "sub_textbook", "reference_material", "prerequisite_material"]
dedup_columns = [col for col in DEDUP_CANDIDATE_COLUMNS if col in df_dedup_base.columns]
missing_dedup_columns = [col for col in DEDUP_CANDIDATE_COLUMNS if col not in df_dedup_base.columns]
if missing_dedup_columns:
    print("[INFO] 중복 기준에서 제외된 미존재 컬럼:", missing_dedup_columns)

rows_before, cols_before = df_dedup_base.shape
duplicate_mask = df_dedup_base.duplicated(subset=dedup_columns, keep="first")
removed_duplicates_log = df_dedup_base.loc[duplicate_mask].copy()
df_step_03 = df_dedup_base.loc[~duplicate_mask].copy()

step_03_path = OUTPUT_DIR / "step_03_duplicate_removed.csv"
duplicates_log_path = OUTPUT_DIR / "removed_duplicates_log.csv"
df_step_03.to_csv(step_03_path, index=False, encoding="utf-8-sig")
removed_duplicates_log.to_csv(duplicates_log_path, index=False, encoding="utf-8-sig")
print(f"Saved: {step_03_path.resolve()}")
print(f"Saved: {duplicates_log_path.resolve()}")

add_log("remove_duplicates", rows_before, len(df_step_03), cols_before, df_step_03.shape[1], f"Removed duplicate syllabus rows using columns={dedup_columns}.")
display(pd.DataFrame({
    "metric": ["rows_before", "rows_after", "removed_duplicate_rows", "removal_ratio"],
    "value": [rows_before, len(df_step_03), int(duplicate_mask.sum()), round(duplicate_mask.sum() / rows_before * 100, 2) if rows_before else 0],
}))
display(df_dedup_base.loc[df_dedup_base.duplicated(subset=dedup_columns, keep=False)].sort_values(dedup_columns).head(50))


## 9. 결측치 기반 품질 정제

추천 품질에 직접적으로 중요한 핵심 컬럼인 `learning_objective`와 `main_textbook` 기준을 강화한다. 두 컬럼은 과목의 학습 목표와 학습 자료를 이해하는 데 중요하므로, 둘 중 하나라도 실질적으로 비어 있으면 제거한다.

실질적인 결측으로 보는 값은 `NaN`, 빈 문자열, 공백만 있는 문자열, `-`, `--`, `/`, `//`, `미정`, `추후공지`, `추후 공지`, `추후공고`, `추후 공고`, `해당없음`, `해당 없음`, `없다`이다.

`sub_textbook`, `reference_material`, `prerequisite_material`은 원래 비어 있을 수 있는 보조 정보이므로 단독 제거 기준으로 사용하지 않는다.

In [ ]:
IMPORTANT_REQUIRED_COLUMNS = ["course_name", "course_type", "college_name", "department_name"]
QUALITY_COLUMNS = ["course_name", "grade", "semester", "credit", "course_type", "learning_objective", "main_textbook", "sub_textbook", "reference_material", "prerequisite_material", "college_name", "department_name"]
CORE_QUALITY_COLUMNS = ["learning_objective", "main_textbook"]
MISSING_LIKE_VALUES = {"", "-", "--", "/", "//", "미정", "추후공지", "추후 공지", "추후공고", "추후 공고", "해당없음", "해당 없음", "없다"}


def is_effectively_missing(series: pd.Series) -> pd.Series:
    normalized = series.fillna("").astype(str).map(normalize_text)
    return normalized.isin(MISSING_LIKE_VALUES)


def is_blank_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.apply(is_effectively_missing)


df_missing_base = df_step_03.copy()
rows_before, cols_before = df_missing_base.shape
existing_required_columns = [col for col in IMPORTANT_REQUIRED_COLUMNS if col in df_missing_base.columns]
existing_quality_columns = [col for col in QUALITY_COLUMNS if col in df_missing_base.columns]
existing_core_quality_columns = [col for col in CORE_QUALITY_COLUMNS if col in df_missing_base.columns]
print("Existing required columns:", existing_required_columns)
print("Existing quality columns:", existing_quality_columns)
print("Existing core quality columns:", existing_core_quality_columns)

blank_required = is_blank_frame(df_missing_base[existing_required_columns]).any(axis=1) if existing_required_columns else pd.Series(False, index=df_missing_base.index)
learning_missing = is_effectively_missing(df_missing_base.get("learning_objective", pd.Series("", index=df_missing_base.index)))
textbook_missing = is_effectively_missing(df_missing_base.get("main_textbook", pd.Series("", index=df_missing_base.index)))
quality_blank_ratio = is_blank_frame(df_missing_base[existing_quality_columns]).mean(axis=1) if existing_quality_columns else pd.Series(0, index=df_missing_base.index)
high_missing_ratio = quality_blank_ratio >= 0.60

missing_reasons = pd.Series("", index=df_missing_base.index, dtype="object")
missing_reasons.loc[blank_required] += "missing_required_metadata;"
missing_reasons.loc[learning_missing] += "missing_learning_objective;"
missing_reasons.loc[textbook_missing] += "missing_main_textbook;"
missing_reasons.loc[high_missing_ratio] += "quality_missing_ratio_60_or_more;"
missing_reasons = missing_reasons.str.rstrip(";")

remove_missing_mask = missing_reasons.ne("")
removed_missing_log = df_missing_base.loc[remove_missing_mask].copy()
removed_missing_log["missing_removal_reason"] = missing_reasons.loc[remove_missing_mask]
removed_missing_log["quality_missing_ratio"] = quality_blank_ratio.loc[remove_missing_mask]

df_step_04 = df_missing_base.loc[~remove_missing_mask].copy()
df_step_04["quality_missing_ratio"] = quality_blank_ratio.loc[~remove_missing_mask]

step_04_path = OUTPUT_DIR / "step_04_missing_cleaned.csv"
missing_log_path = OUTPUT_DIR / "removed_missing_log.csv"
df_step_04.to_csv(step_04_path, index=False, encoding="utf-8-sig")
removed_missing_log.to_csv(missing_log_path, index=False, encoding="utf-8-sig")
print(f"Saved: {step_04_path.resolve()}")
print(f"Saved: {missing_log_path.resolve()}")

add_log(
    "strict_core_quality_filter",
    rows_before,
    len(df_step_04),
    cols_before,
    df_step_04.shape[1],
    "Keep only rows where both learning_objective and main_textbook exist. This improves curriculum-generation quality while avoiding overly strict removal based on optional columns.",
)

core_quality_summary_df = pd.DataFrame({
    "metric": [
        "before_core_quality_filter_rows",
        "after_core_quality_filter_rows",
        "removed_by_core_quality_filter",
        "removed_ratio_by_core_quality_filter",
    ],
    "value": [
        rows_before,
        len(df_step_04),
        int(remove_missing_mask.sum()),
        round(remove_missing_mask.sum() / rows_before * 100, 2) if rows_before else 0,
    ],
})

display(core_quality_summary_df)
display(removed_missing_log["missing_removal_reason"].value_counts().rename_axis("missing_removal_reason").reset_index(name="row_count"))


## 10. 최종 컬럼 정리

최종 데이터에는 커리큘럼 생성과 추적에 필요한 컬럼만 유지한다. `course_type`은 앞선 단계에서 교양 제거와 전공/교양 필터링에 이미 사용되었으므로 최종 추천용 데이터셋에서는 제거한다.

분석용 플래그 컬럼은 제거하지만, `keep_reason`은 유지한다. 최종 데이터에서 왜 해당 행이 살아남았는지 추적하기 위해서다.

In [ ]:
FINAL_COLUMNS = [
    "source_row_number",
    "university_name",
    "college_name",
    "department_name",
    "course_name",
    "grade",
    "semester",
    "credit",
    "learning_objective",
    "main_textbook",
    "sub_textbook",
    "reference_material",
    "prerequisite_material",
    "keep_reason",
]
existing_final_columns = [col for col in FINAL_COLUMNS if col in df_step_04.columns]
missing_final_columns = [col for col in FINAL_COLUMNS if col not in df_step_04.columns]
if missing_final_columns:
    print("[INFO] 최종 컬럼 목록에서 제외된 미존재 컬럼:", missing_final_columns)

rows_before, cols_before = df_step_04.shape
df_final = df_step_04[existing_final_columns].copy()
final_path = OUTPUT_DIR / "university_syllabus_eda_cleaned_final.csv"
df_final.to_csv(final_path, index=False, encoding="utf-8-sig")
print(f"Saved: {final_path.resolve()}")

add_log("select_final_columns", rows_before, len(df_final), cols_before, df_final.shape[1], "Selected final recommendation columns and removed course_type plus analysis-only flags.")

final_columns = list(df_final.columns)
final_column_check_df = pd.DataFrame({
    "metric": ["final_columns", "course_type in final_columns"],
    "value": [", ".join(final_columns), "course_type" in final_columns],
})
display(final_column_check_df)
display(pd.DataFrame({"final_column": final_columns}))


## 11. 최종 검증 EDA

최종 데이터가 실제 추천용 데이터셋으로 사용할 수 있는지 검증한다. 특히 `learning_objective`, `main_textbook` 존재 비율과 `course_type` 제거 여부를 확인한다.

In [ ]:
final_shape_df = pd.DataFrame({"metric": ["rows", "columns"], "value": [df_final.shape[0], df_final.shape[1]]})
final_missing_df = pd.DataFrame({
    "column": df_final.columns,
    "missing_count": df_final.apply(lambda col: is_effectively_missing(col).sum()).values,
    "missing_ratio": (df_final.apply(lambda col: is_effectively_missing(col).mean()) * 100).round(2).values,
})
learning_objective_exists = ~is_effectively_missing(df_final.get("learning_objective", pd.Series("", index=df_final.index)))
main_textbook_exists = ~is_effectively_missing(df_final.get("main_textbook", pd.Series("", index=df_final.index)))
quality_ratio_df = pd.DataFrame({
    "metric": ["learning_objective_exists_ratio", "main_textbook_exists_ratio", "both_learning_objective_and_main_textbook_ratio"],
    "ratio_percent": [round(learning_objective_exists.mean() * 100, 2), round(main_textbook_exists.mean() * 100, 2), round((learning_objective_exists & main_textbook_exists).mean() * 100, 2)],
})

final_validation_df = pd.DataFrame({
    "metric": ["final_shape", "course_type in final_columns"],
    "value": [str(df_final.shape), "course_type" in df_final.columns],
})

display(final_shape_df)
display(final_validation_df)
display(pd.DataFrame({"final_column": df_final.columns}))
display(final_missing_df)
display(quality_ratio_df)
display(df_final.get("college_name", pd.Series(dtype=str)).value_counts().head(30).rename_axis("college_name").reset_index(name="row_count"))
display(df_final.get("department_name", pd.Series(dtype=str)).value_counts().head(50).rename_axis("department_name").reset_index(name="row_count"))
display(df_final.get("course_name", pd.Series(dtype=str)).value_counts().head(100).rename_axis("course_name").reset_index(name="row_count"))
display(df_final.get("grade", pd.Series(dtype=str)).value_counts().rename_axis("grade").reset_index(name="row_count"))
display(df_final.get("keep_reason", pd.Series(dtype=str)).value_counts().rename_axis("keep_reason").reset_index(name="row_count"))
display(df_final.sample(n=min(20, len(df_final)), random_state=42) if len(df_final) else df_final)


## 12. 단계별 산출물 비교

각 단계별 행 수를 비교하여 어느 단계에서 데이터가 얼마나 줄었는지 확인한다. `step_04_missing_cleaned.csv`와 최종 파일에는 강화된 핵심 품질 조건이 반영된다.

In [ ]:
step_outputs = [
    {"step": "raw_data", "file_name": str(source_path), "df": raw_df, "note": "Original source data."},
    {"step": "step_01_target_domain_filtered", "file_name": str(step_01_path), "df": df_step_01, "note": "Strict target-domain rows only."},
    {"step": "step_02_noise_removed", "file_name": str(step_02_path), "df": df_step_02, "note": "General education and obvious noise removed."},
    {"step": "step_03_duplicate_removed", "file_name": str(step_03_path), "df": df_step_03, "note": "Duplicate syllabus content removed."},
    {"step": "step_04_missing_cleaned", "file_name": str(step_04_path), "df": df_step_04, "note": "Strict core quality filter: both learning_objective and main_textbook exist."},
    {"step": "final", "file_name": str(final_path), "df": df_final, "note": "Final columns selected and course_type removed."},
]

summary_records = []
prev_rows = None
for item in step_outputs:
    rows = item["df"].shape[0]
    cols = item["df"].shape[1]
    removed = 0 if prev_rows is None else prev_rows - rows
    removal_ratio = 0 if prev_rows in (None, 0) else round(removed / prev_rows * 100, 2)
    summary_records.append({
        "step": item["step"],
        "file_name": item["file_name"],
        "rows": rows,
        "cols": cols,
        "rows_removed_from_previous": removed,
        "removal_ratio_from_previous": removal_ratio,
        "note": item["note"],
    })
    prev_rows = rows

step_summary_df = pd.DataFrame(summary_records)
step_summary_path = OUTPUT_DIR / "eda_cleaning_step_summary.csv"
step_summary_df.to_csv(step_summary_path, index=False, encoding="utf-8-sig")
print(f"Saved: {step_summary_path.resolve()}")
display(step_summary_df)


## 13. EDA 로그 저장

전체 EDA 과정의 로그를 기존 로그 파일명에 덮어쓴다. 별도의 새 로그 파일은 만들지 않는다.

In [ ]:
eda_cleaning_log_df = pd.DataFrame(eda_logs)
log_path = OUTPUT_DIR / "eda_cleaning_log.csv"
eda_cleaning_log_df.to_csv(log_path, index=False, encoding="utf-8-sig")
print(f"Saved: {log_path.resolve()}")
display(eda_cleaning_log_df)

print("\nUpdated CSV files:")
for path in [
    step_01_path,
    step_02_path,
    step_03_path,
    step_04_path,
    final_path,
    duplicates_log_path,
    missing_log_path,
    step_summary_path,
    log_path,
]:
    print(path.resolve())


In [ ]:
# 10. RAG 청킹 전략 수립을 위한 텍스트 분석
# Purpose: 정제된 강의계획서 데이터의 주요 텍스트 필드를 분석해 RAG 청킹 기준을 정합니다.
# Input: df_final_quality 또는 저장된 최종/중간 산출물
# Output: rag_field_summary_df, rag_length_distribution_df, rag_chunking_strategy_df, rag_sample_df

import math
import re
import unicodedata
from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 220)

def normalize_for_rag(value):
    """한글 자모 분리와 과도한 공백을 줄여 RAG 분석용 텍스트를 안정화합니다."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_syllabus_for_rag():
    for var_name in ["df_final_quality", "df_final_fixed", "df_final_eda", "df_cs_ee_clean"]:
        if var_name in globals():
            return globals()[var_name].copy(), f"memory: {var_name}"

    candidates = [
        Path("outputs/final/05_syllabus_final_quality_filtered.csv"),
        Path("outputs/final/04_syllabus_metadata_restored_deduplicated.csv"),
        Path("outputs/intermediate/03_syllabus_deduplicated.csv"),
        Path("outputs/intermediate/02_syllabus_domain_filtered.csv"),
    ]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, encoding="utf-8-sig"), str(path)
    raise FileNotFoundError("RAG 분석용 syllabus 최종/중간 산출물을 찾을 수 없습니다.")

rag_courses_df, rag_source = load_syllabus_for_rag()
rag_courses_df = rag_courses_df.copy()

text_fields = [
    col for col in [
        "course_name", "learning_objective", "main_textbook", "sub_textbook",
        "reference_material", "prerequisite_material", "course_type",
        "college_name", "department_name", "grade", "semester", "credit"
    ]
    if col in rag_courses_df.columns
]

for col in text_fields:
    rag_courses_df[col] = rag_courses_df[col].map(normalize_for_rag)

if "learning_objective" not in rag_courses_df.columns:
    raise KeyError("RAG 청킹 분석에 필요한 learning_objective 컬럼이 없습니다.")

content_fields = [
    col for col in [
        "course_name", "learning_objective", "main_textbook", "sub_textbook",
        "reference_material", "prerequisite_material"
    ]
    if col in rag_courses_df.columns
]
metadata_fields = [col for col in text_fields if col not in content_fields]

rag_courses_df["rag_text"] = rag_courses_df[content_fields].agg(
    lambda row: "\n".join(f"{col}: {row[col]}" for col in content_fields if row[col]),
    axis=1,
)

analysis_fields = [col for col in content_fields + ["rag_text"] if col in rag_courses_df.columns]

summary_rows = []
for col in analysis_fields:
    series = rag_courses_df[col].fillna("").map(normalize_for_rag)
    lengths = series.str.len()
    non_empty = lengths > 0
    summary_rows.append({
        "field": col,
        "rows": len(series),
        "non_empty_rows": int(non_empty.sum()),
        "empty_rows": int((~non_empty).sum()),
        "mean_chars": round(lengths[non_empty].mean(), 1) if non_empty.any() else 0,
        "median_chars": round(lengths[non_empty].median(), 1) if non_empty.any() else 0,
        "p75_chars": round(lengths[non_empty].quantile(0.75), 1) if non_empty.any() else 0,
        "p90_chars": round(lengths[non_empty].quantile(0.90), 1) if non_empty.any() else 0,
        "p95_chars": round(lengths[non_empty].quantile(0.95), 1) if non_empty.any() else 0,
        "max_chars": int(lengths.max()) if len(lengths) else 0,
    })

rag_field_summary_df = pd.DataFrame(summary_rows)

bins = [0, 100, 300, 500, 1000, 2000, 5000, np.inf]
labels = ["~100", "101~300", "301~500", "501~1000", "1001~2000", "2001~5000", "5000+"]
distribution_rows = []
for col in analysis_fields:
    lengths = rag_courses_df[col].fillna("").map(normalize_for_rag).str.len()
    bucket = pd.cut(lengths, bins=bins, labels=labels, include_lowest=True)
    counts = bucket.value_counts().sort_index()
    for label, count in counts.items():
        distribution_rows.append({"field": col, "length_bucket": str(label), "count": int(count)})

rag_length_distribution_df = pd.DataFrame(distribution_rows)

rag_text_lengths = rag_courses_df["rag_text"].str.len()
recommended_chunk_chars = int(min(1500, max(600, math.ceil(rag_text_lengths.quantile(0.90) / 100) * 100)))
recommended_overlap_chars = int(round(recommended_chunk_chars * 0.15 / 10) * 10)
long_record_threshold = recommended_chunk_chars * 1.5
long_record_count = int((rag_text_lengths > long_record_threshold).sum())

rag_chunking_strategy_df = pd.DataFrame([
    {
        "dataset": "curriculum_courses",
        "source": rag_source,
        "document_unit": "1 row = 1 course syllabus",
        "primary_text": "learning_objective + textbooks/material fields",
        "metadata_fields": ", ".join(metadata_fields),
        "recommended_chunk_chars": recommended_chunk_chars,
        "recommended_overlap_chars": recommended_overlap_chars,
        "split_rule": "강의 단위 유지, 교재/참고자료가 긴 행만 줄바꿈/문장 기준으로 분할",
        "long_record_threshold_chars": int(long_record_threshold),
        "long_record_count": long_record_count,
    }
])

rag_sample_columns = [col for col in ["course_name", "college_name", "department_name", "learning_objective", "main_textbook"] if col in rag_courses_df.columns]
rag_sample_df = (
    rag_courses_df.assign(rag_text_chars=rag_text_lengths)[rag_sample_columns + ["rag_text_chars"]]
    .sort_values("rag_text_chars", ascending=False)
    .head(5)
)

print("=" * 70)
print("RAG 청킹 전략 분석 - curriculum_courses")
print("=" * 70)
print(f"데이터 출처: {rag_source}")
print(f"행 수: {len(rag_courses_df):,}")
print("\n[필드별 텍스트 길이 요약]")
display(rag_field_summary_df)
print("\n[길이 구간 분포]")
display(rag_length_distribution_df.pivot(index="length_bucket", columns="field", values="count").fillna(0).astype(int))
print("\n[권장 청킹 전략]")
display(rag_chunking_strategy_df)
print("\n[긴 문서 샘플]")
display(rag_sample_df)
